# Diagnóstico inicial de calidad de datos — FiveM Analytics

**Proyecto:** Análisis de rendimiento, retención de jugadores y economía virtual en servidores de roleplay de FiveM GTA V.

**Pregunta analítica:** ¿Cuáles son las actividades, trabajos y zonas del mapa con mayor concurrencia de jugadores dentro del servidor de FiveM, y cómo influye el balance económico (ingresos vs. gastos virtuales) en la retención y tiempo promedio de juego diario de la comunidad?

Por defecto, inspecciona nueve CSV sintéticos sobre un escenario FiveM (sesiones, jugadores, economía, vehículos, actividad y rendimiento) y un dataset público de Kaggle. El archivo de Kaggle también fue creado sintéticamente por su autor; ninguna fuente es telemetría observada de un servidor real. Los identificadores son ficticios y se redactan en las muestras. Cambia `DATA_MODE` a `"real"` solo cuando agregues exportaciones auténticas autorizadas y anonimizadas en `data/raw/`. El Notebook diagnostica y resume el escenario, no limpia ni corrige automáticamente.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

SENSITIVE_NAME_PARTS = (
    "steam", "discord", "license", "citizenid", "citizen_id", "token",
    "player_id", "playerid", "user_id", "userid", "identifier", "username", "plate",
    "session_id", "event_id", "transaction_id", "vehicle_id",
)

def is_sensitive_column(column: object) -> bool:
    normalized = re.sub(r"[^a-z0-9]+", "_", str(column).lower())
    return any(part in normalized for part in SENSITIVE_NAME_PARTS)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_MODE = "synthetic"  # Usa "real" cuando tengas ambas fuentes auténticas.

if DATA_MODE == "synthetic":
    SOURCE_DIRS = {
        "FiveM sintético (demostración)": PROJECT_ROOT / "data" / "examples" / "fivem",
        "Kaggle (dataset público sintético)": PROJECT_ROOT / "data" / "examples" / "kaggle",
    }
    print("AVISO: FiveM es un escenario sintético; el dataset público de Kaggle también se declara sintético por su autor.")
elif DATA_MODE == "real":
    SOURCE_DIRS = {
        "FiveM / MySQL / txAdmin": PROJECT_ROOT / "data" / "raw" / "fivem",
        "Kaggle / telemetría y retención": PROJECT_ROOT / "data" / "raw" / "kaggle",
    }
else:
    raise ValueError("DATA_MODE debe ser 'synthetic' o 'real'.")

SUPPORTED_SUFFIXES = {".csv"}

print(f"Raíz del proyecto: {PROJECT_ROOT}")
for source, folder in SOURCE_DIRS.items():
    print(f"{source}: {folder}")

In [ ]:
def list_source_files(folder: Path) -> list[Path]:
    """List supported tabular files recursively in a source folder."""
    if not folder.is_dir():
        raise FileNotFoundError(f"No existe la carpeta de la fuente: {folder}")
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    )


def load_tabular_file(path: Path) -> pd.DataFrame:
    """Load one CSV file and report errors with its path."""
    if path.suffix.lower() == ".csv":
        columns = pd.read_csv(path, encoding="utf-8-sig", nrows=0).columns
        identifier_types = {
            column: "string" for column in columns if is_sensitive_column(column)
        }
        return pd.read_csv(path, encoding="utf-8-sig", dtype=identifier_types)
    raise ValueError(f"Formato no admitido para {path}; usa CSV.")


source_files = {
    source: list_source_files(folder)
    for source, folder in SOURCE_DIRS.items()
}
missing_sources = [source for source, files in source_files.items() if not files]
if missing_sources:
    expected = "\n".join(f"- {source}: {SOURCE_DIRS[source]}" for source in missing_sources)
    raise FileNotFoundError(
        "No se encontraron archivos CSV para todas las fuentes del proyecto. "
        "Agrega los CSV requeridos; fuentes sin archivos:\n" + expected
    )

dataframes = {}
for source, files in source_files.items():
    for path in files:
        relative_path = path.relative_to(SOURCE_DIRS[source]).with_suffix("")
        name = f"{source} / {relative_path.as_posix()}"
        dataframes[name] = load_tabular_file(path)

print(f"Archivos cargados correctamente: {len(dataframes)}")
for name, dataframe in dataframes.items():
    print(f"- {name}: {dataframe.shape[0]} filas × {dataframe.shape[1]} columnas")

## Inspección estructural

Cada archivo se conserva en un DataFrame independiente. Para limitar la exposición accidental de identificadores directos en las muestras y descripciones categóricas, se redactan columnas cuyos nombres sugieren Steam ID, Discord ID, license, citizen ID, token o claves de jugador/usuario. Esto no sustituye la anonimización del archivo de origen.

In [ ]:
def redacted_view(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Return a display copy with likely direct identifiers masked."""
    result = dataframe.copy()
    for column in result.columns:
        if is_sensitive_column(column):
            result[column] = "[redactado]"
    return result


structure_rows = []
missing_rows = []
for name, dataframe in dataframes.items():
    print(f"\n## {name}")
    print("head() — primeros registros (vista redactada):")
    display(redacted_view(dataframe.head()))
    print("tail() — últimos registros (vista redactada):")
    display(redacted_view(dataframe.tail()))
    print("shape:", dataframe.shape)
    print("columns:", dataframe.columns.tolist())
    print("dtypes:")
    display(dataframe.dtypes.rename("dtype").to_frame())
    print("info():")
    dataframe.info()
    numeric_columns = [
        column for column in dataframe.select_dtypes(include="number").columns
        if not is_sensitive_column(column)
    ]
    print("describe() — variables numéricas no identificadoras:")
    display(dataframe[numeric_columns].describe() if numeric_columns else pd.DataFrame())

    object_columns = [
        column for column in dataframe.columns
        if pd.api.types.is_object_dtype(dataframe[column].dtype)
        or pd.api.types.is_string_dtype(dataframe[column].dtype)
    ]
    if len(object_columns):
        text_data = redacted_view(dataframe[object_columns])
        if all(pd.api.types.is_object_dtype(dataframe[column].dtype) for column in object_columns):
            print("describe(include='object') — variables categóricas (vista redactada):")
            display(text_data.describe(include="object"))
        else:
            print("describe() — variables de texto (vista redactada):")
            display(text_data.describe())
    else:
        print("No hay columnas de tipo object para describe(include='object').")

    structure_rows.append({
        "Dataset": name,
        "Filas": dataframe.shape[0],
        "Columnas": dataframe.shape[1],
        "Tipos de datos": ", ".join(sorted(map(str, dataframe.dtypes.unique()))),
    })
    null_counts = dataframe.isna().sum()
    for column, count in null_counts[null_counts.gt(0)].items():
        missing_rows.append({
            "Dataset": name,
            "Columna": str(column),
            "Valores faltantes": int(count),
            "Porcentaje": round(float(count / len(dataframe) * 100), 2) if len(dataframe) else 0.0,
        })

structure = pd.DataFrame(structure_rows)
missing_summary = pd.DataFrame(
    missing_rows,
    columns=["Dataset", "Columna", "Valores faltantes", "Porcentaje"],
)
display(structure)
print("Valores faltantes por dataset y columna:")
display(missing_summary if not missing_summary.empty else pd.DataFrame({"Resultado": ["No se observaron valores faltantes en los archivos cargados."]}))

## Indicadores iniciales y lectura crítica

El conteo de duplicados siguiente identifica filas completamente iguales; no prueba que una fila repetida sea un error. Confirma claves y reglas de negocio antes de proponer su eliminación. La inspección tampoco demuestra por sí sola que las ubicaciones, duraciones, fechas o movimientos económicos sean semánticamente válidos.

In [ ]:
NUMERIC_RULES = {
    "duration_minutes": (1, None),
    "duration_min": (1, 720),
    "amount": (-1_000_000, 1_000_000),
    "online_players": (0, 128),
    "players_online": (0, 128),
    "max_players": (1, None),
    "server_fps": (1, 100),
    "tickrate": (1, 60),
    "cpu_percent": (0, 100),
    "cpu_pct": (0, 100),
    "memory_percent": (0, 100),
    "ram_mb": (0, None),
    "cash": (0, None),
    "bank": (0, 100_000_000),
    "age_character": (18, 100),
    "banned": (0, 1),
    "price_paid": (0, None),
    "fuel": (0, 100),
    "Age": (0, None),
    "PlayTimeHours": (0, None),
    "InGamePurchases": (0, 1),
    "SessionsPerWeek": (0, None),
    "AvgSessionDurationMinutes": (0, None),
    "PlayerLevel": (0, None),
    "AchievementsUnlocked": (0, None),
}
DATE_COLUMNS = {
    "connected_at", "disconnected_at", "transaction_at", "event_at", "snapshot_at",
    "connect_time", "disconnect_time", "timestamp", "registered_at", "last_seen",
    "purchase_date",
}
quality_rows = []
finding_rows = []

for name, dataframe in dataframes.items():
    duplicate_count = int(dataframe.duplicated().sum())
    quality_rows.append({
        "Dataset": name,
        "Filas duplicadas exactas": duplicate_count,
        "Columnas con faltantes": int(dataframe.isna().any().sum()),
        "Columnas numéricas": int(len(dataframe.select_dtypes(include="number").columns)),
        "Columnas texto": sum(
            pd.api.types.is_object_dtype(dataframe[column].dtype)
            or pd.api.types.is_string_dtype(dataframe[column].dtype)
            for column in dataframe.columns
        ),
    })
    if duplicate_count:
        finding_rows.append({
            "Dataset": name, "Variable": "Fila completa", "Problema": "Duplicado exacto",
            "Registros": duplicate_count, "Evidencia/regla": "dataframe.duplicated()",
        })

    for column in dataframe.columns:
        missing_count = int(dataframe[column].isna().sum())
        if missing_count:
            finding_rows.append({
                "Dataset": name, "Variable": str(column), "Problema": "Valor faltante",
                "Registros": missing_count, "Evidencia/regla": "isna()",
            })

        if column in NUMERIC_RULES:
            numeric = pd.to_numeric(dataframe[column], errors="coerce")
            non_numeric_count = int((dataframe[column].notna() & numeric.isna()).sum())
            if non_numeric_count:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Tipo/valor no numérico",
                    "Registros": non_numeric_count, "Evidencia/regla": "to_numeric(errors='coerce')",
                })
            minimum, maximum = NUMERIC_RULES[column]
            outside = pd.Series(False, index=dataframe.index)
            if minimum is not None:
                outside |= numeric.lt(minimum)
            if maximum is not None:
                outside |= numeric.gt(maximum)
            outside_count = int(outside.sum())
            if outside_count:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Fuera del rango orientador",
                    "Registros": outside_count,
                    "Evidencia/regla": f"mínimo={minimum}; máximo={maximum}",
                })

        if pd.api.types.is_object_dtype(dataframe[column].dtype) or pd.api.types.is_string_dtype(dataframe[column].dtype):
            text_values = dataframe[column].astype("string")
            trimmed_values = text_values.str.strip()
            whitespace_count = int((text_values.notna() & text_values.ne(trimmed_values)).sum())
            if whitespace_count:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Espacios al inicio/final",
                    "Registros": whitespace_count, "Evidencia/regla": "valor distinto tras strip()",
                })
            normalized = trimmed_values.str.casefold()
            variants = pd.DataFrame({"normalizado": normalized, "original": text_values}).dropna()
            variant_keys = variants.groupby("normalizado")["original"].nunique()
            variant_keys = variant_keys[variant_keys.gt(1)].index
            variant_count = int(
                (text_values.notna() & normalized.isin(variant_keys) & text_values.ne(normalized)).sum()
            )
            if variant_count:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Variantes de mayúsculas/minúsculas",
                    "Registros": variant_count, "Evidencia/regla": "valores distintos tras strip().casefold()",
                })

        if column in DATE_COLUMNS:
            parsed_dates = pd.to_datetime(dataframe[column], errors="coerce")
            invalid_dates = int((dataframe[column].notna() & parsed_dates.isna()).sum())
            if invalid_dates:
                finding_rows.append({
                    "Dataset": name, "Variable": str(column), "Problema": "Fecha/hora no interpretable",
                    "Registros": invalid_dates, "Evidencia/regla": "to_datetime(errors='coerce')",
                })

    if {"connected_at", "disconnected_at"}.issubset(dataframe.columns):
        connected = pd.to_datetime(dataframe["connected_at"], errors="coerce")
        disconnected = pd.to_datetime(dataframe["disconnected_at"], errors="coerce")
        reversed_sessions = int((connected.notna() & disconnected.notna() & disconnected.lt(connected)).sum())
        if reversed_sessions:
            finding_rows.append({
                "Dataset": name, "Variable": "connected_at/disconnected_at",
                "Problema": "Desconexión anterior a conexión", "Registros": reversed_sessions,
                "Evidencia/regla": "disconnected_at < connected_at",
            })

def find_table(table_name: str):
    return next(
        (frame for name, frame in dataframes.items()
         if name.rsplit("/", 1)[-1].strip().lower() == table_name.lower()),
        None,
    )

players_table = find_table("players")
if players_table is not None and "player_id" in players_table:
    player_key = players_table["player_id"].astype("string")
    duplicate_keys = int(player_key.duplicated().sum())
    if duplicate_keys:
        finding_rows.append({
            "Dataset": next(name for name, frame in dataframes.items() if frame is players_table),
            "Variable": "player_id", "Problema": "Clave primaria duplicada",
            "Registros": duplicate_keys, "Evidencia/regla": "player_id.duplicated()",
        })
    valid_player_ids = set(player_key.dropna())
    for table_name, foreign_key in (
        ("sessions", "player_id"),
        ("transactions", "player_id"),
        ("vehicles", "owner_player_id"),
    ):
        child_table = find_table(table_name)
        if child_table is None or foreign_key not in child_table:
            continue
        foreign_values = child_table[foreign_key].astype("string")
        orphan_count = int((foreign_values.notna() & ~foreign_values.isin(valid_player_ids)).sum())
        if orphan_count:
            child_name = next(name for name, frame in dataframes.items() if frame is child_table)
            finding_rows.append({
                "Dataset": child_name, "Variable": foreign_key,
                "Problema": "Referencia sin jugador correspondiente", "Registros": orphan_count,
                "Evidencia/regla": "clave ausente en players.player_id",
            })

performance_table = find_table("server_performance")
if performance_table is not None and "timestamp" in performance_table:
    observed_times = pd.to_datetime(performance_table["timestamp"], errors="coerce").dropna()
    if len(observed_times) > 1:
        expected_times = pd.date_range(observed_times.min(), observed_times.max(), freq="h")
        missing_intervals = len(expected_times.difference(pd.DatetimeIndex(observed_times)))
        if missing_intervals:
            performance_name = next(name for name, frame in dataframes.items() if frame is performance_table)
            finding_rows.append({
                "Dataset": performance_name, "Variable": "timestamp",
                "Problema": "Huecos en muestreo horario esperado", "Registros": missing_intervals,
                "Evidencia/regla": "horas esperadas menos marcas observadas",
            })

quality_summary = pd.DataFrame(quality_rows)
findings = pd.DataFrame(
    finding_rows,
    columns=["Dataset", "Variable", "Problema", "Registros", "Evidencia/regla"],
)
display(quality_summary)
print("Hallazgos automáticos para revisión; los rangos son reglas orientadoras, no reglas universales del dominio:")
display(findings if not findings.empty else pd.DataFrame({"Resultado": ["No se detectaron condiciones mediante las reglas ejecutadas."]}))

print("\nInterpretación requerida antes de preparar los datos:")
print("1. Contrastar fechas, unidades y definiciones de variables con la documentación de cada fuente.")
print("2. Validar las reglas de rango orientadoras contra el esquema y el significado de cada fuente.")
print("3. Distinguir faltantes legítimos de errores de captura y justificar el tratamiento por variable.")
print("4. Confirmar periodo, población y licencia del dataset de Kaggle antes de compararlo con FiveM.")
print("5. Registrar en reports/informe_diagnostico.md la evidencia, impacto y acción propuesta por hallazgo.")

## Resumen analítico del escenario ampliado

Estas tablas descriptivas relacionan las fuentes sintéticas por sus claves simuladas. El bloque excluye referencias huérfanas y duraciones/importes fuera de los límites orientadores para los cálculos señalados; conserva intactos los DataFrames originales. La repetición a 7 días es solo una métrica proxy del escenario, no una medida real de retención. Las correlaciones y rankings no son evidencia de un servidor real ni implican causalidad.

In [ ]:
players_data = find_table("players")
sessions_data = find_table("sessions")
transactions_data = find_table("transactions")
vehicles_data = find_table("vehicles")
performance_data = find_table("server_performance")

required_tables = {
    "players": players_data,
    "sessions": sessions_data,
    "transactions": transactions_data,
    "vehicles": vehicles_data,
    "server_performance": performance_data,
}
missing_tables = [name for name, frame in required_tables.items() if frame is None]
if missing_tables:
    print("No se calculó el resumen analítico ampliado; faltan estas tablas:", ", ".join(missing_tables))
else:
    players_clean_key = players_data.drop_duplicates("player_id", keep="first").copy()
    known_player_ids = set(players_clean_key["player_id"].dropna().astype("string"))

    session_analysis = sessions_data.copy()
    session_analysis["duration_numeric"] = pd.to_numeric(session_analysis["duration_min"], errors="coerce")
    session_analysis["connect_datetime"] = pd.to_datetime(session_analysis["connect_time"], errors="coerce")
    linked_sessions = session_analysis[
        session_analysis["player_id"].astype("string").isin(known_player_ids)
    ].copy()
    valid_sessions = linked_sessions[
        linked_sessions["duration_numeric"].between(1, 720)
        & linked_sessions["connect_datetime"].notna()
    ].copy()
    print(f"Jugadores únicos para el análisis: {len(players_clean_key):,}")
    print(f"Sesiones enlazadas a jugadores: {len(linked_sessions):,} de {len(session_analysis):,}")
    print(f"Sesiones con duración entre 1 y 720 minutos y fecha válida: {len(valid_sessions):,}")

    activity_summary = valid_sessions.groupby("main_activity").agg(
        sesiones=("session_id", "size"),
        duracion_mediana_min=("duration_numeric", "median"),
    ).sort_values("sesiones", ascending=False).round(1)
    print("Sesiones por actividad (duración mediana solo con dato numérico):")
    display(activity_summary)

    zone_summary = valid_sessions.groupby("zone", dropna=False).agg(
        sesiones=("session_id", "size"),
        duracion_mediana_min=("duration_numeric", "median"),
    ).sort_values("sesiones", ascending=False).round(1)
    print("Sesiones por zona (la ubicación sintética no representa un mapa observado):")
    display(zone_summary)

    first_session = valid_sessions.groupby("player_id")["connect_datetime"].min()
    observation_end = valid_sessions["connect_datetime"].max()
    eligible_ids = first_session[first_session <= observation_end - pd.Timedelta(days=7)].index
    seven_day_cutoffs = first_session.loc[eligible_ids] + pd.Timedelta(days=7)
    returned_ids = set(
        valid_sessions.loc[
            valid_sessions["player_id"].isin(eligible_ids)
            & valid_sessions.apply(
                lambda row: row["connect_datetime"] >= seven_day_cutoffs.get(row["player_id"], pd.Timestamp.max),
                axis=1,
            ),
            "player_id",
        ]
    )
    repeat_rate = len(returned_ids) / len(eligible_ids) if len(eligible_ids) else float("nan")
    print(
        "Proxy de repetición a 7 días: "
        f"{len(returned_ids):,}/{len(eligible_ids):,} jugadores elegibles ({repeat_rate:.1%}); "
        "no equivale a retención real."
    )

    transaction_analysis = transactions_data.copy()
    transaction_analysis["type_normalized"] = transaction_analysis["type"].astype("string").str.strip().str.casefold()
    transaction_analysis["amount_numeric"] = pd.to_numeric(
        transaction_analysis["amount"].astype("string").str.replace(r"[$,]", "", regex=True),
        errors="coerce",
    )
    valid_amounts = transaction_analysis["amount_numeric"].abs().le(1_000_000)
    transaction_summary = transaction_analysis[valid_amounts].groupby("type_normalized").agg(
        transacciones=("transaction_id", "size"),
        importe_mediano=("amount_numeric", "median"),
        suma_firmada=("amount_numeric", "sum"),
    ).sort_values("transacciones", ascending=False).round(2)
    print(
        "Transacciones por tipo tras normalizar espacios/mayúsculas y excluir importes > 1,000,000 "
        "(regla exploratoria):"
    )
    display(transaction_summary)
    print(
        "Importes no interpretables o fuera de ese umbral: "
        f"{int((~valid_amounts).sum()):,} de {len(transaction_analysis):,}."
    )

    player_sessions = valid_sessions.groupby("player_id")["duration_numeric"].sum().rename("minutes_played")
    player_transactions = transaction_analysis[
        transaction_analysis["player_id"].astype("string").isin(known_player_ids)
        & valid_amounts
    ].groupby("player_id")["amount_numeric"].sum().rename("signed_amount")
    player_comparison = pd.concat([player_sessions, player_transactions], axis=1).dropna()
    correlation = player_comparison["minutes_played"].corr(player_comparison["signed_amount"])
    print(
        "Correlación descriptiva entre minutos válidos y suma firmada de transacciones "
        f"(jugadores emparejados n={len(player_comparison):,}): {correlation:.3f}. "
        "No demuestra causalidad y depende de los límites exploratorios usados."
    )

    vehicle_summary = vehicles_data.groupby("model").agg(
        vehiculos=("vehicle_id", "size"),
        precio_mediano=("price_paid", "median"),
        combustible_mediano=("fuel", "median"),
    ).sort_values("vehiculos", ascending=False).round(1)
    print("Vehículos por modelo (descripción del escenario):")
    display(vehicle_summary)

    performance_analysis = performance_data.copy()
    for column in ("players_online", "server_fps", "cpu_pct", "tickrate", "ram_mb"):
        performance_analysis[column] = pd.to_numeric(performance_analysis[column], errors="coerce")
    plausible_performance = performance_analysis[
        performance_analysis["players_online"].between(0, 128)
        & performance_analysis["server_fps"].between(1, 60)
        & performance_analysis["cpu_pct"].between(0, 100)
    ]
    fps_correlation = plausible_performance[["players_online", "server_fps"]].corr().iloc[0, 1]
    print(
        "Correlación descriptiva jugadores concurrentes/FPS en rangos orientadores "
        f"(n={len(plausible_performance):,}): {fps_correlation:.3f}. No es una medición real."
    )
    resource_summary = performance_analysis.groupby("heaviest_resource").agg(
        muestras=("timestamp", "size"),
        fps_promedio=("server_fps", "mean"),
        cpu_promedio=("cpu_pct", "mean"),
    ).sort_values("muestras", ascending=False).round(2)
    print("Métricas agrupadas por etiqueta de recurso simulada; no atribuye causalidad:")
    display(resource_summary)

## Cierre

Este Notebook realiza un diagnóstico inicial, no una limpieza completa ni un análisis causal. Los conteos y resúmenes impresos deben trasladarse al informe junto con una interpretación basada en el dominio. Antes de guardar o publicar el Notebook, revisa y borra cualquier salida que pudiera exponer datos sensibles.